In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
BASE_DIR

WindowsPath('c:/Users/Daisy/Desktop/all desktop/Deskstop/Thesis/Daisy/slob-forecasting')

In [3]:
DATA_DIR = BASE_DIR / 'data'
DATA_DIR.exists()

True

In [4]:
SLOB_DATASET = DATA_DIR / 'slob.csv'
SLOB_DATASET.exists()

True

In [5]:
date_cols = ['Week', 'Date_First_Listed']

In [6]:
slob = pd.read_csv(SLOB_DATASET, parse_dates=date_cols)
print(f"Shape: {slob.shape[0]:,} rows x {slob.shape[1]} columns")

Shape: 290,450 rows x 26 columns


In [7]:
slob.head()

,Week,SKU,Product_Category,ABC,Date_First_Listed,Time_to_Expiry_Days,Units_Sold,Stock_Level,Lead_Time,Safety_Stock,...,Non_Food_CPI,Inflation_Rate,Rolling_4W_Avg,Rolling_8W_Avg,ADI,CV2,Zero_Demand_Ratio,Consec_Zero_Weeks,Inventory_Turnover,SLOB_Label
0,2023-01-02,Amul_Biscuits_0166,Ambient Grocery Foods,A,2023-02-07,180,287,511,8.2,336,...,297.43,6.4,287.00,287.00,1.0329,0.1103,0.0318,0,0.5616,0
1,2023-01-09,Amul_Biscuits_0166,Ambient Grocery Foods,A,2023-02-07,180,303,633,8.2,346,...,297.43,6.4,295.00,295.00,1.0329,0.1103,0.0318,0,0.4787,0
2,2023-01-16,Amul_Biscuits_0166,Ambient Grocery Foods,A,2023-02-07,180,260,632,8.2,332,...,297.43,6.4,283.33,283.33,1.0329,0.1103,0.0318,0,0.4114,0
3,2023-01-23,Amul_Biscuits_0166,Ambient Grocery Foods,A,2023-02-07,180,306,568,8.2,339,...,297.43,6.4,289.00,289.00,1.0329,0.1103,0.0318,0,0.5387,0
4,2023-01-30,Amul_Biscuits_0166,Ambient Grocery Foods,A,2023-02-07,180,227,461,8.2,321,...,297.43,6.4,274.00,276.60,1.0329,0.1103,0.0318,0,0.4924,0


In [8]:
slob.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 290450 entries, 0 to 290449
Data columns (total 26 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Week                 290450 non-null  datetime64[us]
 1   SKU                  290450 non-null  str           
 2   Product_Category     290450 non-null  str           
 3   ABC                  290450 non-null  str           
 4   Date_First_Listed    290450 non-null  datetime64[us]
 5   Time_to_Expiry_Days  290450 non-null  int64         
 6   Units_Sold           290450 non-null  int64         
 7   Stock_Level          290450 non-null  int64         
 8   Lead_Time            290450 non-null  float64       
 9   Safety_Stock         290450 non-null  int64         
 10  Reorder_Level        290450 non-null  int64         
 11  Price                290450 non-null  float64       
 12  Promotion            290450 non-null  int64         
 13  Seasonal_Index       2904

In [9]:
slob_clean = slob.copy()

Basic cleanup

In [10]:
# categorical typing
categorical_cols = ['SKU', 'Product_Category', 'ABC']
for col in categorical_cols:
    slob_clean[col] = slob_clean[col].astype('category')

In [11]:
# handling duplicates
duplicated = slob_clean.duplicated(subset=['SKU', 'Week']).sum()
duplicated

np.int64(0)

In [12]:
# canonical sort
slob_clean = slob_clean.sort_values(['SKU', 'Week']).reset_index(drop=True)
slob_clean.dtypes

Week                   datetime64[us]
SKU                          category
Product_Category             category
ABC                          category
Date_First_Listed      datetime64[us]
Time_to_Expiry_Days             int64
Units_Sold                      int64
Stock_Level                     int64
Lead_Time                     float64
Safety_Stock                    int64
Reorder_Level                   int64
Price                         float64
Promotion                       int64
Seasonal_Index                float64
Overall_CPI                   float64
Food_CPI                      float64
Non_Food_CPI                  float64
Inflation_Rate                float64
Rolling_4W_Avg                float64
Rolling_8W_Avg                float64
ADI                           float64
CV2                           float64
Zero_Demand_Ratio             float64
Consec_Zero_Weeks               int64
Inventory_Turnover            float64
SLOB_Label                      int64
dtype: objec

In [13]:
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.exists()

True

In [14]:
ARTIFACTS_DATA_DIR = ARTIFACTS_DIR / 'data'
ARTIFACTS_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DATA_DIR.exists()

True

In [15]:
slob_clean.to_parquet(ARTIFACTS_DATA_DIR / 'slob.parquet', index=False, engine='pyarrow')